In [ ]:
import requests
r = requests.get("https://proverki.gov.ru/", timeout=20)
print(r.status_code)

In [ ]:
s = requests.Session()
s.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "ru-RU,ru;q=0.9",
    "Referer": "https://proverki.gov.ru/portal/public-inspections",
    "Origin": "https://proverki.gov.ru",
})

s.get("https://proverki.gov.ru/portal", timeout=20)          # забрать куки
r = s.get("https://proverki.gov.ru/public/api/inspections/find",
          params={"page": "50,1", "searchString": "7707083893"}, timeout=20)
print(r.status_code, r.text[:300])

In [ ]:
!pip install httpx[http2]

In [ ]:
import httpx
with httpx.Client(http2=True, headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "ru-RU,ru;q=0.9",
    "Referer": "https://proverki.gov.ru/portal/public-inspections",
    "Origin": "https://proverki.gov.ru"}, timeout=20) as c:
    r = c.get("https://proverki.gov.ru/portal")
    print(r.status_code)

In [ ]:
print(r)

In [ ]:
import httpx
from time import sleep

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "ru-RU,ru;q=0.9",
    "Referer": "https://proverki.gov.ru/portal/public-inspections",
}

BASE = "https://proverki.gov.ru/public/api/inspections/find"
INN = "7707083893"

with httpx.Client(http2=True, headers=HEADERS, timeout=20, follow_redirects=True) as c:
    c.get("https://proverki.gov.ru/portal")          # взять куки

    tries = [
        ("raw comma",      f"{BASE}?page=50,1&searchString={INN}", None),
        ("encoded comma",  f"{BASE}?page=50%2C1&searchString={INN}", None),
        ("params dict",    BASE, {"page": "50,1", "searchString": INN}),
        ("reversed pair",  BASE, {"page": "1,50", "searchString": INN}),
        ("page+size",      BASE, {"page": 0, "size": 50, "searchString": INN}),
        ("only search",    BASE, {"searchString": INN}),
    ]

    for name, url, params in tries:
        sleep(2)
        r = c.get(url, params=params)
        body = r.text[:250].replace("\n", " ")
        print(f"{name:15} {r.status_code}  {body}")
        print(dict(r.headers))
        print('-'*80)

raw comma       400  
{'server': 'nginx', 'date': 'Sun, 20 Sep 2026 08:16:35 GMT', 'content-length': '0', 'connection': 'keep-alive', 'server-timing': 'dtSInfo;desc="1"'}
------------------------------
encoded comma   400  
{'server': 'nginx', 'date': 'Sun, 20 Sep 2026 08:16:37 GMT', 'content-length': '0', 'connection': 'keep-alive', 'server-timing': 'dtSInfo;desc="0", dtRpid;desc="-21741765"'}
------------------------------
params dict     400  
{'server': 'nginx', 'date': 'Sun, 20 Sep 2026 08:16:39 GMT', 'content-length': '0', 'connection': 'keep-alive', 'server-timing': 'dtSInfo;desc="1"'}
------------------------------
reversed pair   400  
{'server': 'nginx', 'date': 'Sun, 20 Sep 2026 08:16:41 GMT', 'content-length': '0', 'connection': 'keep-alive', 'server-timing': 'dtSInfo;desc="0", dtRpid;desc="1169458953"'}
------------------------------
page+size       400  
{'server': 'nginx', 'date': 'Sun, 20 Sep 2026 08:16:43 GMT', 'content-length': '0', 'connection': 'keep-alive', 'server

In [10]:
import requests

# ВАЖНО: Сначала получите cookie, зайдя на главную страницу
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://proverki.gov.ru/",
    "Origin": "https://proverki.gov.ru",
    "Sec-Fetch-Dest": "empty",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Site": "same-origin",
    "Connection": "keep-alive",
})

# Заходим на главную, чтобы получить session cookies
session.get("https://proverki.gov.ru/")

# Теперь делаем запрос к API с правильными параметрами
# Скорее всего, page — это номер страницы (с 0 или 1), а size — размер
url = "https://proverki.gov.ru/public/api/inspections/find"
params = {
    "page": 1,            # или 1, попробуйте оба
    "size": 50,           # или другой размер
    "searchString": "7707083893"
}

r = session.get(url, params=params)
print(r.status_code)
print(r.text[:500])

400



мне нужно получать данные из МСП по ИНН, на сайте когда я вбил ИНН в поиск и получил данные, в теле POST запроса ответом на который является нужный мне JSON я не вижу эндпоинта, то есть мой клик просто постит запрос https://rmsp.nalog.ru/search-proc.json, хотя в ответе все нужные мне данные, как мне тогда получать эти данные?